In [ ]:
from glob import glob
from os.path import *
import numpy as np
import matplotlib.pyplot as plt
import sys
from skimage.filters import threshold_otsu
import seaborn as sns

# Add project root to path
sys.path.append(r"/home/luizluz/Documentos/multi-task-fcn")

from src.io_operations import read_tiff
from src.utils import from_255_to_1


In [ ]:
probabilities_map = glob(r"/home/luizluz/Documentos/multi-task-fcn/bioflore_data/v04/iter_001/*/raster_prediction/join_prob_0.8.TIF")

class_map = glob(r"/home/luizluz/Documentos/multi-task-fcn/bioflore_data/v04/iter_001/*/raster_prediction/join_class_0.8.TIF")

In [ ]:
class_map

In [ ]:
probabilities_map

In [ ]:
# Parameters
sample_size = 10_000  # Number of pixels to sample from each region
random_seed = 42

# Initialize lists to store concatenated data
all_classes = []
all_probabilities = []

# Process each region
for region_idx in range(len(class_map)):
    print(f"Processing region {region_idx + 1}/{len(class_map)}")
    
    # Load maps for this region
    class_map_array = read_tiff(class_map[region_idx])
    prob_map_array = read_tiff(probabilities_map[region_idx])
    # Add 1 to class indices (0,1,2,3 -> 1,2,3,4)
    class_map_array = class_map_array + 1
    
    
    # Normalize probability map if needed (from 0-255 to 0-1)
    if prob_map_array.max() > 1.0:
        prob_map_array = from_255_to_1(prob_map_array)
    
    # Ensure same shape
    assert class_map_array.shape == prob_map_array.shape, \
        f"Shape mismatch in region {region_idx}: {class_map_array.shape} vs {prob_map_array.shape}"
    
    # Flatten arrays
    classes_flat = class_map_array.flatten()
    probs_flat = prob_map_array.flatten()
    
    # Remove invalid pixels (e.g., background/zero class if needed)
    # Adjust this mask based on your data (you might want to keep all pixels)
    valid_mask = classes_flat > 0  # Include all classes (1,2,3,4)
    classes_valid = classes_flat[valid_mask]
    probs_valid = probs_flat[valid_mask]
    
    
    # Sample pixels (same positions for both maps)
    if len(classes_valid) > sample_size:
        np.random.seed(random_seed + region_idx)
        sample_indices = np.random.choice(len(classes_valid), sample_size, replace=False)
        classes_sample = classes_valid[sample_indices]
        probs_sample = probs_valid[sample_indices]
    else:
        classes_sample = classes_valid
        probs_sample = probs_valid
    
    # Concatenate to global arrays
    all_classes.append(classes_sample)
    all_probabilities.append(probs_sample)
    
    print(f"  Sampled {len(classes_sample)} pixels")

# Concatenate all regions
all_classes = np.concatenate(all_classes)
all_probabilities = np.concatenate(all_probabilities)

print(f"\nTotal pixels: {len(all_classes)}")
print(f"Unique classes: {np.unique(all_classes)}")

In [ ]:
# Create histograms for each class
unique_classes = np.unique(all_classes)
n_classes = len(unique_classes)

# Store Otsu thresholds
otsu_thresholds = {}

# Calculate number of rows/cols for subplots
n_cols = min(3, n_classes)
n_rows = (n_classes + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
if n_classes == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for idx, class_val in enumerate(unique_classes):
    # Get probabilities for this class
    class_probs = all_probabilities[all_classes == class_val]
    
    # Calculate Otsu threshold
    try:
        otsu_thresh = threshold_otsu(class_probs)
        otsu_thresholds[class_val] = otsu_thresh
    except ValueError:
        # If Otsu fails (e.g., all values are the same), use mean as fallback
        otsu_thresh = np.mean(class_probs)
        otsu_thresholds[class_val] = otsu_thresh
        print(f"Warning: Otsu threshold calculation failed for class {class_val}, using mean instead")
    
    # Create histogram
    ax = axes[idx]
    
    sns.histplot(class_probs, bins=20, kde=True, ax=ax, alpha=0.7, edgecolor='black')
    ax.set_title(f'Class {class_val} (n={len(class_probs)})')
    ax.set_xlabel('Probability')
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)
    
    # Add statistics
    mean_prob = np.mean(class_probs)
    median_prob = np.median(class_probs)
    ax.axvline(mean_prob, color='red', linestyle='--', label=f'Mean: {mean_prob:.3f}')
    ax.axvline(median_prob, color='green', linestyle='--', label=f'Median: {median_prob:.3f}')
    ax.axvline(otsu_thresh, color='blue', linestyle='--', linewidth=2, label=f'Otsu: {otsu_thresh:.3f}')
    ax.legend()

# Hide unused subplots
for idx in range(n_classes, len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.show()

# Print Otsu thresholds
print("\nOtsu thresholds by class:")
print("-" * 40)
for class_val, thresh in otsu_thresholds.items():
    print(f"Class {class_val}: {thresh:.4f}")

In [ ]:
# Summary statistics by class
# Calculate Otsu thresholds if not already calculated
if 'otsu_thresholds' not in globals():
    otsu_thresholds = {}
    for class_val in unique_classes:
        class_probs = all_probabilities[all_classes == class_val]
        try:
            otsu_thresh = threshold_otsu(class_probs)
            otsu_thresholds[class_val] = otsu_thresh
        except ValueError:
            otsu_thresh = np.mean(class_probs)
            otsu_thresholds[class_val] = otsu_thresh

print("Summary statistics by class:")
print("-" * 60)
for class_val in unique_classes:
    class_probs = all_probabilities[all_classes == class_val]
    otsu_thresh = otsu_thresholds.get(class_val, None)
    print(f"\nClass {class_val}:")
    print(f"  Count: {len(class_probs)}")
    print(f"  Mean: {np.mean(class_probs):.4f}")
    print(f"  Median: {np.median(class_probs):.4f}")
    if otsu_thresh is not None:
        print(f"  Otsu threshold: {otsu_thresh:.4f}")
    print(f"  Std: {np.std(class_probs):.4f}")
    print(f"  Min: {np.min(class_probs):.4f}")
    print(f"  Max: {np.max(class_probs):.4f}")
    print(f"  25th percentile: {np.percentile(class_probs, 25):.4f}")
    print(f"  75th percentile: {np.percentile(class_probs, 75):.4f}")

In [ ]:
import geopandas as gpd
gdf = gpd.read_file("/home/luizluz/Documentos/multi-task-fcn/bioflore_data/shapes/labels.shp")

gdf.head()

In [ ]:
gdf["label"].unique()